# Tutorial 1: Basic Prompting with Claude

This tutorial introduces you to the fundamentals of prompting Claude using the Anthropic API.

## What You'll Learn

- Setting up the Anthropic Python SDK
- Making your first API call
- Understanding the Messages API structure
- Best practices for clear, effective prompts
- Model selection and parameters

## Prerequisites

- Python 3.7+
- An Anthropic API key (get one at https://console.anthropic.com/)

---

## Setup: Install the Anthropic SDK

First, install the Anthropic Python SDK:

In [ ]:
!pip install anthropic python-dotenv

## Configure Your API Key

**Security Best Practice:** Never hardcode API keys in your code. Use environment variables instead.

Create a `.env` file in your working directory:
```
ANTHROPIC_API_KEY=your_api_key_here
```

Or set it in your current session:

In [ ]:
import os
from anthropic import Anthropic

# Option 1: Load from .env file
from dotenv import load_dotenv
load_dotenv()

# Option 2: Set directly (for testing only - not recommended for production)
# os.environ["ANTHROPIC_API_KEY"] = "your_key_here"

# Initialize the Anthropic client
client = Anthropic()

print("✓ Anthropic client initialized successfully!")

## Example 1: Your First Claude API Call

Let's make a simple API call to Claude:

In [ ]:
# Make a simple API call
message = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "Hello, Claude! Please introduce yourself in one sentence."}
    ]
)

# Display the response
print("Claude's response:")
print(message.content[0].text)

### Understanding the API Response

Let's examine the full response structure:

In [ ]:
import json

# Display full response metadata
print("Response ID:", message.id)
print("Model:", message.model)
print("Role:", message.role)
print("Stop reason:", message.stop_reason)
print("\nToken usage:")
print(f"  Input tokens: {message.usage.input_tokens}")
print(f"  Output tokens: {message.usage.output_tokens}")
print(f"\nContent blocks: {len(message.content)}")
print(f"Content type: {message.content[0].type}")

## Example 2: Clear vs Vague Instructions

**Pattern:** Clear, Explicit, Direct Instructions

One of the most important prompting principles is being clear and specific. Let's compare:

In [ ]:
# ❌ Vague prompt
vague_response = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=500,
    messages=[{
        "role": "user",
        "content": "Tell me about Python."
    }]
)

print("VAGUE PROMPT RESPONSE:")
print(vague_response.content[0].text)
print(f"\n(Used {vague_response.usage.output_tokens} tokens)")

In [ ]:
# ✅ Clear, specific prompt
clear_response = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=500,
    messages=[{
        "role": "user",
        "content": """Explain the three main use cases for Python list comprehensions.
For each use case:
1. Provide a code example
2. Show the equivalent for-loop version
3. Explain when to use it

Keep each example under 3 lines of code."""
    }]
)

print("CLEAR PROMPT RESPONSE:")
print(clear_response.content[0].text)
print(f"\n(Used {clear_response.usage.output_tokens} tokens)")

### Key Takeaway

The clear prompt produces more useful, structured output. Notice how:
- It specifies exactly what format we want
- It sets constraints (3 examples, under 3 lines each)
- It defines the structure (code + explanation)

**Impact:** Clear instructions can improve response accuracy by ~30% (source: patterns/prompting/clear-instructions.md)

## Example 3: Using System Prompts

System prompts set the context and behavior for the entire conversation:

In [ ]:
# Using a system prompt to set Claude's role
response = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=300,
    system="You are a senior Python developer specializing in code review. You provide concise, actionable feedback focused on best practices, performance, and readability.",
    messages=[{
        "role": "user",
        "content": """Review this code:

```python
def get_data(ids):
    results = []
    for id in ids:
        results.append(database.query(id))
    return results
```"""
    }]
)

print(response.content[0].text)

## Example 4: Model Selection

Choose the right model for your use case:

| Model | Best For | Speed | Cost |
|-------|----------|-------|------|
| claude-3-5-sonnet-20241022 | Complex tasks, coding | Medium | Medium |
| claude-3-5-haiku-20241022 | Fast responses, simple tasks | Fast | Low |
| claude-3-opus-20240229 | Most sophisticated reasoning | Slow | High |

Let's compare response times:

In [ ]:
import time

models = [
    "claude-3-5-haiku-20241022",
    "claude-3-5-sonnet-20241022"
]

prompt = "Explain what a REST API is in one sentence."

for model in models:
    start_time = time.time()
    
    response = client.messages.create(
        model=model,
        max_tokens=100,
        messages=[{"role": "user", "content": prompt}]
    )
    
    elapsed = time.time() - start_time
    
    print(f"\n{'='*60}")
    print(f"Model: {model}")
    print(f"Response time: {elapsed:.2f}s")
    print(f"Tokens: {response.usage.output_tokens}")
    print(f"\nResponse: {response.content[0].text}")

## Example 5: Controlling Output with Temperature

The `temperature` parameter (0.0 to 1.0) controls randomness:
- **0.0**: Deterministic, focused responses
- **1.0**: Creative, varied responses

In [ ]:
prompt = "Generate a creative name for a coffee shop."

# Low temperature (deterministic)
response_low = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=50,
    temperature=0.0,
    messages=[{"role": "user", "content": prompt}]
)

# High temperature (creative)
response_high = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=50,
    temperature=1.0,
    messages=[{"role": "user", "content": prompt}]
)

print("Temperature = 0.0 (Focused):")
print(response_low.content[0].text)
print("\nTemperature = 1.0 (Creative):")
print(response_high.content[0].text)

## Example 6: Prefilling for Format Control

**Pattern:** Response Prefilling

You can prefill Claude's response to control the format. This achieves near 100% format consistency:

In [ ]:
# Without prefilling - format may vary
response_normal = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=150,
    messages=[{
        "role": "user",
        "content": "What is 25 * 4? Respond with just the number."
    }]
)

print("Without prefilling:")
print(response_normal.content[0].text)

# With prefilling - guaranteed format
response_prefilled = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=150,
    messages=[
        {"role": "user", "content": "What is 25 * 4?"},
        {"role": "assistant", "content": "{\"result\": "}  # Prefill JSON format
    ]
)

print("\nWith prefilling:")
print("{\"result\": " + response_prefilled.content[0].text)

## Interactive Exercise: Build Your Own Prompt

Now it's your turn! Modify the prompt below to:
1. Ask Claude to explain a concept of your choice
2. Specify the format you want
3. Set constraints (length, style, etc.)

In [ ]:
# YOUR TURN: Modify this prompt
your_prompt = """[Replace this with your prompt]

Format:
- [Specify format]

Constraints:
- [Add constraints]
"""

response = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=500,
    messages=[{"role": "user", "content": your_prompt}]
)

print(response.content[0].text)

## Key Takeaways

1. **Be Clear and Specific**: Vague prompts get vague responses. Specify format, length, and structure.

2. **Use System Prompts**: Set the context and role for the entire conversation.

3. **Choose the Right Model**: 
   - Haiku for speed and simple tasks
   - Sonnet for balanced performance
   - Opus for maximum capability

4. **Control Randomness**: Use temperature (0.0 = focused, 1.0 = creative)

5. **Prefill for Consistency**: Start Claude's response to guarantee format

## Next Steps

- **Tutorial 2**: Learn how to stream responses for better user experience
- **Tutorial 3**: Explore tool use and function calling
- **Tutorial 4**: Master conversation management and context
- **Tutorial 5**: Apply production patterns for real-world applications

## Resources

- [Anthropic API Documentation](https://docs.anthropic.com/)
- [Prompting Guide](../patterns/prompting/clear-instructions.md)
- [Anthropic Official Patterns](../patterns/prompting/anthropic-official.md)